In [ ]:
import requests, time, re, pickle, warnings
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from tqdm import tqdm
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout, Bidirectional
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
from tensorflow.keras.utils import to_categorical

warnings.filterwarnings('ignore')
print("✅ TensorFlow:", tf.__version__)
print("✅ All imports done!")


In [ ]:
API_KEY  = "310dfba3252d94966cb65a063f8174058108"
BASE_URL = "https://api.fda.gov/drug/label.json"

drug_terms = [
    "ibuprofen", "aspirin", "paracetamol",
    "metformin", "amoxicillin", "insulin",
    "omeprazole", "atorvastatin", "azithromycin",
    "cetirizine", "diclofenac", "prednisone", "salbutamol"
]
print(f"📋 {len(drug_terms)} drugs:", drug_terms)


In [ ]:
def fetch_medical_data(terms, target_size=2000):
    all_data = []
    print("🚀 Collecting data...")
    while len(all_data) < target_size:
        for term in tqdm(terms, desc="Fetching"):
            url = f"{BASE_URL}?search=openfda.generic_name:{term}&limit=100"
            r   = requests.get(url)
            if r.status_code != 200:
                continue
            for item in r.json().get("results", []):
                all_data.append({
                    "drug"    : term,
                    "text"    : " ".join(item.get("description", [""])),
                    "purpose" : " ".join(item.get("purpose",     [""])),
                    "warnings": " ".join(item.get("warnings",    [""]))
                })
                if len(all_data) >= target_size:
                    break
            time.sleep(0.2)
            if len(all_data) >= target_size:
                break
    df = pd.DataFrame(all_data)
    print(f"\n✅ Collected: {len(df)} records")
    print(df["drug"].value_counts())
    return df

df = fetch_medical_data(drug_terms, target_size=2000)
df.head(3)


In [ ]:
def clean_text(text):
    text = str(text).lower()
    text = re.sub(r"[^a-zA-Z\s]", " ", text)
    text = re.sub(r"\s+",         " ", text)
    return text.strip()

df["clean_text"] = (
    df["text"].fillna("") + " " +
    df["purpose"].fillna("") + " " +
    df["warnings"].fillna("")
)
df["clean_text"] = df["clean_text"].apply(clean_text)
df = df[df["clean_text"].str.len() > 10].reset_index(drop=True)

print(f"✅ After cleaning: {len(df)} records")
print("\nSample:")
print(df["clean_text"].iloc[0][:300])


In [ ]:
from sklearn.preprocessing import LabelEncoder

VOCAB_SIZE    = 5000
MAX_LEN       = 120
EMBEDDING_DIM = 64

# Label encoding
le = LabelEncoder()
df["label"] = le.fit_transform(df["drug"])
NUM_CLASSES  = len(le.classes_)
print(f"✅ {NUM_CLASSES} classes:", list(le.classes_))

# Save the label encoder
with open("label_encoder.pkl", "wb") as f:
    pickle.dump(le, f)
print("✅ label_encoder.pkl saved!")

# Tokenize
tokenizer = Tokenizer(num_words=VOCAB_SIZE, oov_token="<OOV>")
tokenizer.fit_on_texts(df["clean_text"])
sequences = tokenizer.texts_to_sequences(df["clean_text"])

X = pad_sequences(sequences, maxlen=MAX_LEN, padding="post", truncating="post")
y = to_categorical(df["label"], num_classes=NUM_CLASSES)

print(f"\n✅ X shape: {X.shape}")
print(f"✅ y shape: {y.shape}")

In [ ]:
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, random_state=42, stratify=y)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=42, stratify=y_temp)

print(f"Train : {X_train.shape[0]}")
print(f"Val   : {X_val.shape[0]}")
print(f"Test  : {X_test.shape[0]}")


In [ ]:

import os, re, pickle, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from tensorflow.keras.models import Sequential, load_model
from tensorflow.keras.layers import (Embedding, GRU, Dense,
                                      Dropout, Bidirectional)
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
from tensorflow.keras.utils import to_categorical
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

warnings.filterwarnings("ignore")

PROJECT_DIR = '/content/drive/MyDrive/AI_Medical_Assistant'
MODELS_DIR  = f'{PROJECT_DIR}/models'
DATA_DIR    = f'{PROJECT_DIR}/data'
GRU_FILE    = f'{MODELS_DIR}/gru_drug_classifier.h5'

print("✅ TF:", tf.__version__)


In [ ]:

sequences = tokenizer.texts_to_sequences(df['clean_text'])
X = pad_sequences(sequences, maxlen=MAX_LEN, padding='post', truncating='post')
y = to_categorical(df['label'], num_classes=NUM_CLASSES)

X_train,X_temp,y_train,y_temp = train_test_split(X,y,test_size=.30,random_state=42,stratify=y)
X_val,X_test,y_val,y_test     = train_test_split(X_temp,y_temp,test_size=.50,random_state=42,stratify=y_temp)

print(f"✅ Classes ({NUM_CLASSES}): {list(le.classes_)}")
print(f"✅ Train:{X_train.shape[0]} | Val:{X_val.shape[0]} | Test:{X_test.shape[0]}")

In [ ]:
def build_gru(vocab_size, embedding_dim, max_len, num_classes):
    model = Sequential([
        Embedding(vocab_size, embedding_dim, input_length=max_len, name="embedding"),
        Bidirectional(GRU(128, return_sequences=True), name="biGRU_1"),
        Dropout(0.3),
        GRU(64, name="GRU_2"),
        Dropout(0.3),
        Dense(64, activation="relu"),
        Dropout(0.2),
        Dense(num_classes, activation="softmax", name="output")
    ], name="GRU_Drug_Classifier")

    model.compile(optimizer="adam",
                  loss="categorical_crossentropy",
                  metrics=["accuracy"])
    return model

if os.path.exists(GRU_FILE):
    print("✅ GRU found on Drive → loading...")
    gru_model = load_model(GRU_FILE)
else:
    print("⏳ Training GRU...")
    LOCAL_BEST = '/content/best_gru.h5'

    gru_model = build_gru(VOCAB_SIZE, EMBEDDING_DIM, MAX_LEN, NUM_CLASSES)

    history = gru_model.fit(
        X_train, y_train,
        validation_data=(X_val, y_val),
        epochs=10,
        batch_size=32,
        callbacks=[
            EarlyStopping(monitor='val_accuracy', patience=5,
                          restore_best_weights=True, verbose=1),
            ModelCheckpoint(LOCAL_BEST, monitor='val_accuracy',
                            save_best_only=True, verbose=1)
        ]
    )
    gru_model = load_model(LOCAL_BEST)
    gru_model.save(GRU_FILE)
    print(f"✅ Saved to Drive: {GRU_FILE}")

In [ ]:
try:
    fig, axes = plt.subplots(1, 2, figsize=(13, 4))
    axes[0].plot(history.history['accuracy'],     label='Train', color='purple')
    axes[0].plot(history.history['val_accuracy'], label='Val',   color='tomato')
    axes[0].set_title('GRU Accuracy'); axes[0].legend(); axes[0].grid(alpha=.3)

    axes[1].plot(history.history['loss'],     label='Train', color='purple')
    axes[1].plot(history.history['val_loss'], label='Val',   color='tomato')
    axes[1].set_title('GRU Loss'); axes[1].legend(); axes[1].grid(alpha=.3)

    plt.suptitle('GRU Training Curves', fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.savefig(f'{PROJECT_DIR}/gru_training.png', dpi=150)
    plt.show()
except:
    print("(Model loaded from Drive — no history to plot)")


In [ ]:
loss, acc = gru_model.evaluate(X_test, y_test, verbose=0)
y_pred     = gru_model.predict(X_test, verbose=0)
y_pred_cls = np.argmax(y_pred, axis=1)
y_true_cls = np.argmax(y_test, axis=1)

print(f"\n{'='*50}")
print(f"📊 GRU Results")
print(f"{'='*50}")
print(f"🎯 Test Accuracy : {acc*100:.2f}%")
print(f"📉 Test Loss     : {loss:.4f}")
print(f"\n{classification_report(y_true_cls, y_pred_cls, target_names=le.classes_)}")

cm = confusion_matrix(y_true_cls, y_pred_cls)
plt.figure(figsize=(11, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Purples',
            xticklabels=le.classes_, yticklabels=le.classes_)
plt.title('Confusion Matrix — GRU', fontsize=13)
plt.ylabel('True'); plt.xlabel('Predicted')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig(f'{PROJECT_DIR}/gru_confusion.png', dpi=150)
plt.show()

gru_acc = acc


In [ ]:
y_pred_gru = gru_model.predict(X_test, verbose=0)
gru_pred = np.argmax(y_pred_gru, axis=1)
gru_true = np.argmax(y_test, axis=1)
print("✅ gru_pred and gru_true created.")

In [ ]:
loss, acc = gru_model.evaluate(X_test, y_test, verbose=0)
y_pred     = gru_model.predict(X_test, verbose=0)
y_pred_cls = np.argmax(y_pred, axis=1)
y_true_cls = np.argmax(y_test, axis=1)

print(f"\n{'='*50}")
print(f"📊 GRU Model Evaluation on Test Set")
print(f"{'='*50}")
print(f"🎯 Test Accuracy : {acc*100:.2f}%")
print(f"📉 Test Loss     : {loss:.4f}")
print(f"\n{classification_report(y_true_cls, y_pred_cls, target_names=le.classes_)}")

In [ ]:
def predict_drug_gru(text, top_k=3):
    cleaned = clean_text(text)
    seq     = tokenizer.texts_to_sequences([cleaned])
    padded  = pad_sequences(seq, maxlen=MAX_LEN, padding='post', truncating='post')
    probs   = gru_model.predict(padded, verbose=0)[0]
    top_idx = np.argsort(probs)[::-1][:top_k]

    print(f'\n[GRU] 🔍 "{text[:65]}"')
    print('-' * 42)
    for i, idx in enumerate(top_idx, 1):
        conf = probs[idx] * 100
        bar  = '█' * int(conf/5) + '░' * (20 - int(conf/5))
        print(f'  {i}. {le.classes_[idx]:<16} {conf:>5.1f}%  {bar}')
    print('-' * 42)
    return le.classes_[top_idx[0]]

# ── Tests ──
predict_drug_gru('used for pain relief fever reduction anti inflammatory')
predict_drug_gru('antibiotic bacterial infection throat ear')
predict_drug_gru('controls blood sugar type 2 diabetes')
predict_drug_gru('asthma bronchospasm inhaler bronchodilator')
